In [1]:
import medmnist
from medmnist import OCTMNIST
from medmnist import INFO
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision.models import resnet18
from torchvision import models
from torchvision import transforms
from sklearn.model_selection import train_test_split
from utils.dataset import Dataset 
from dotenv import load_dotenv
import os

In [2]:
load_dotenv() # Carica le variabili d'ambiente dal file .env (device gpu e cpu)


data_flag = 'octmnist'
download_flag = True

device = os.getenv('DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

info = INFO[data_flag] #Ottieni le informazioni specifiche del dataset (es. task, n_channels, n_classes, python_class)
task = info['task'] #tipo di task (es. 'multi-class', 'binary', 'multi-label')
n_channels = info['n_channels'] #numero di canali dell'immagine (es. 3 per RGB, 1 per grayscale)
n_classes = len(info['label']) #numero di classi (es. 10 per cifar10, 2 per binary classification)

DataClass = getattr(medmnist, info['python_class']) #Ottieni la classe Python corrispondente al dataset (es. CIFAR10, MNIST) usando il nome specificato in info['python_class']

Using device: cuda


PREPROCESS STANDARD (IMAGENET) + ESCLUSIONE FULLY CONNECTED LAYER

In [3]:
backbone = resnet18(weights=models.ResNet18_Weights.DEFAULT)
backbone.fc = torch.nn.Identity() #Rimuoviamo l'ultimo strato fully connected per ottenere solo le features estratte dal backbone
backbone.eval() #Mettiamo il modello in modalità eval per disabilitare dropout e batch normalization (se presenti) durante l'estrazione delle features

weights = models.ResNet18_Weights.DEFAULT #Riassegniamo i pesi di default per accedere alle statistiche di normalizzazione
preprocess = weights.transforms() # Otteniamo le trasformazioni di preprocessing predefinite per ResNet18, che includono la normalizzazione con media e deviazione standard specifiche
print(preprocess.mean)


# Definisci le trasformazioni con la conversione RGB in cima
preprocess = transforms.Compose([
    # 1. Converte l'immagine PIL da scala di grigio a RGB (3 canali identici)
    transforms.Lambda(lambda img: img.convert('RGB')), 
    
    # 2. Converte in tensore PyTorch
    transforms.ToTensor(),
    
    # 3. Ora la normalizzazione funzionerà perfettamente perché il tensore ha 3 canali
    transforms.Normalize(preprocess.mean, preprocess.std)
])


[0.485, 0.456, 0.406]


DOWNLOAD DATASET (con flag su medmnist)

In [ ]:
data_flag = "octmnist"
download_flag = False  # Cambia a True se vuoi scaricare il dataset (solo la prima volta)

# Load MedMNIST datasets directly - NO concatenation!
train_dataset = DataClass(
     split="train", transform=preprocess, download=download_flag, size=224
)
val_dataset = DataClass(
     split="val", transform=preprocess, download=download_flag, size=224
)
test_dataset = DataClass(
     split="test", transform=preprocess, download=download_flag, size=224
)     

print(f"Train Dataset: {len(train_dataset)} images")
print(f"Val Dataset: {len(val_dataset)} images")
print(f"Test Dataset: {len(test_dataset)} images")
data_toprocess = {
    "train": train_dataset,
    "val": val_dataset,
    "test": test_dataset
}

Train Dataset: 97477 images
Val Dataset: 10832 images
Test Dataset: 1000 images


ESTRAZIONE FEATURE DAL DATASET --> Visual Embeddings R 512 (vettore) e costruzione
dataset di feature


In [ ]:
# Helper function to extract features from a dataset without loading everything to memory
extraction_flag = False

# Funzione per estrarre features e labels da un dataset in batch, evitando di caricare tutto in memoria
def extract_features_from_dataset(dataset, backbone, batch_size=32, device='cpu'):
    """
    Extract features and labels from a dataset in batches.
    Returns numpy arrays of features and labels.
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    features_list = []
    labels_list = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            
            # Extract features using backbone
            pooled = backbone(imgs)
            flat = pooled.view(pooled.size(0), -1)  # (batch, 512)
            
            # Move back to CPU for storage
            features_list.append(flat.cpu().numpy())
            labels_list.append(labels.numpy())
    
    # Concatenate all batches (handles variable batch sizes, including incomplete final batch)
    all_features = np.concatenate(features_list, axis=0)  # Shape: (num_samples, 512)
    all_labels = np.concatenate(labels_list, axis=0).flatten()  # Shape: (num_samples,)
    
    return all_features, all_labels


# Set device (GPU if available)
print(f"Using device: {device}")

backbone_device = backbone.to(device)

dataframe = pd.DataFrame()

if extraction_flag:
 for dataset_name, dataset in data_toprocess.items():
    print(f"\n{'='*60}")
    print(f"Processing Dataset: {dataset_name}")
    print(f"{'='*60}")
    
    features, labels = extract_features_from_dataset(
        dataset, backbone_device, batch_size=32, device=device
    )
    
    print(f"{dataset_name.capitalize()} Features: {features.shape}, Labels: {labels.shape}")
    dataset = pd.DataFrame(features)
    dataset['label'] = labels
    dataframe = pd.concat([dataframe, dataset], ignore_index=True)

Using device: cuda


In [ ]:
print("\nFinal DataFrame shape:", dataframe.shape)
#save_path = os.path.join(os.getcwd(), "feature_extracted_dataset", "dataset_features_and_labels.csv")
#dataset_feature_and_labels.to_csv(save_path, index=False)

#Da salvare in pytorch, molto piu veloce, si riconverte con torch.load() 
# e non si perde nulla in termini di precisione o struttura dei dati
torch.save(dataframe, os.path.join(os.getcwd(), "feature_extracted_dataset", "dataset_features_and_labels.pt"))


Final DataFrame shape: (109309, 513)


In [ ]:
del train_dataset, val_dataset, test_dataset  # Free memory after processing datasets

In [ ]:
# Carica il dataset con features estratte e labels, se extraction_flag è False, 
# altrimenti si utilizza il dataframe appena creato

dataset_feature_and_labels = torch.load(os.path.join(os.getcwd(), "feature_extracted_dataset", "dataset_features_and_labels.pt"),weights_only=False)

In [12]:
dataset_feature_and_labels.shape

(109309, 513)

In [ ]:
from pathlib import Path
# Extract features for each dataset split
seeds = [11, 17, 29]
seeds_flag= True
if not seeds_flag:
 for seed in seeds:
    print(f"\n{'='*60}")
    print(f"Processing Seed: {seed}")
    print(f"{'='*60}")
    

    # Lo split stratificato in train, val e test, corrisponde a 80% train, 10% val e 10% test, 
    # mantenendo la stessa distribuzione delle classi in ogni split
    train, val_test = train_test_split(dataset_feature_and_labels,
                                        test_size=0.2, random_state=seed, stratify=dataset_feature_and_labels['label'])
    val, test = train_test_split(val_test,
                                  test_size=0.5, random_state=seed, stratify=val_test['label'])
    


    root = Path.cwd()
    print(root)  # controlla la cartella di lavoro
    path = root / "assets" / f"seeds_{seed}"
     # Save each split separately to CSV
    print(f"\n--- Saving Features to CSV for Seed: {seed} ---")
    
    os.makedirs(path, exist_ok=True)
    train.to_csv(f"{path}/train_{seed}.csv", index=False)
    val.to_csv(f"{path}/val_{seed}.csv", index=False)
    test.to_csv(f"{path}/test_{seed}.csv", index=False)